# Notebook 01 - Data Cleaning 

In this notebook we clean the data of the dataframe obtained in the notebook $\tt 00\_data\_scraping.ipynb$.

In [1]:

# General
import numpy as np                     
import pandas as pd   
import re              

# Plotting Options
from matplotlib import pyplot as plt                           
import seaborn as sns                                          
plt.rcParams['text.usetex'] = True                             
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'  

# File management
from pathlib import Path                  
import os                                 

# defines a function to find the project root directory by looking for a specific marker (default is 'data')
def find_project_root(start=Path.cwd(), marker='data'): 
    current = start
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root() 
os.chdir(PROJECT_ROOT) # go to the root of the project 

In [91]:
# importing dataframe 

df_reviews = pd.read_csv('data/dataframes/bronze/all_kindle_review.csv')
df_reviews.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


Let us get some general informations regarding our dataframe so we know what to clean

In [92]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Unnamed: 0.1    12000 non-null  int64 
 1   Unnamed: 0      12000 non-null  int64 
 2   asin            12000 non-null  object
 3   helpful         12000 non-null  object
 4   rating          12000 non-null  int64 
 5   reviewText      12000 non-null  object
 6   reviewTime      12000 non-null  object
 7   reviewerID      12000 non-null  object
 8   reviewerName    11962 non-null  object
 9   summary         11998 non-null  object
 10  unixReviewTime  12000 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 1.0+ MB


We note that there are null values in the reviwerName column. We could remove those reviewers, for example, but for our analysis this will not be necessary since we are only interested in the reviews and this is what we will use to train our model. We drop one of the columns that came with the dataframe and rename the other which is a unique id. 

In [93]:
df_reviews = df_reviews.drop(columns=['Unnamed: 0.1'])
df_reviews = df_reviews.rename(columns={'Unnamed: 0': 'unique_id'})

In [94]:
df_reviews.head()

,unique_id,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


We also change the reviewTime column to date format

In [95]:
df_reviews['normalized_review_time'] = pd.to_datetime(df_reviews['reviewTime'].astype(str),errors='coerce')

For simplicity, we will create a function that normalizes the review text so we minimze potential problems that may appear

In [96]:
def text_normalize(text):
    
    """
    Function to clean the text:
    1. Converts to lower case.
    2. Removes ponctuation, numbers and special characters.
    3. Removes extra spaces.
    """
    
    # Guarantees that the text in non-null (in case of any NaN in the DataFrame)
    if not isinstance(text, str):
        return ""

    # Converter para minúsculas
    clean_text = text.lower()
    
    # Manter apenas letras e espaços. A remoção de acentos já foi feita.
    clean_text = re.sub(r'[^a-z\s]', '', clean_text)
    
    # Remover espaços extras
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    
    return clean_text

Checking if the function works correctly

In [97]:
review_test = df_reviews.loc[1,'reviewText']
print(review_test)

Great short read.  I didn't want to put it down so I read it all in one sitting.  The sex scenes were great between the two males and one female character...a bit surprising - I never thought you could do that!  I learned something new and really enjoyed reading this book!  This is a great way to get all hot and bothered and take advantage of your significant other(s)!


In [98]:
text_normalize(review_test)

'great short read i didnt want to put it down so i read it all in one sitting the sex scenes were great between the two males and one female charactera bit surprising i never thought you could do that i learned something new and really enjoyed reading this book this is a great way to get all hot and bothered and take advantage of your significant others'

Since it works as we desire, let us apply to the entire review column

In [99]:
df_reviews['normalized_review'] = df_reviews['reviewText'].apply(text_normalize)

In [100]:
df_reviews.head()

,unique_id,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime,normalized_review_time,normalized_review
0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600,2010-09-02,jace rankin may be short but hes nothing to me...
1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400,2013-10-08,great short read i didnt want to put it down s...
2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400,2014-04-11,ill start by saying this is the first of four ...
3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400,2014-07-05,aggie is angela lansbury who carries pocketboo...
4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000,2012-12-31,i did not expect this type of book to be in li...


To simplify our project and analysis, we will just classify the reviews as "positive" or "negative". The negative reviews will be defined as the ones with rating 1 or 2 and will be mapped to the number 0. The positive reviews will be defined as the ones with rating 4 or 5 and will be mapped to the number 1. The reviews with rating 3 will be dropped for simplicity. 

In [101]:
# removing line with rating = 3 
df_reviews = df_reviews[df_reviews['rating'] != 3]

# maping ratings 1 and 2 into 0 
df_reviews['rating'] = df_reviews['rating'].replace([1, 2], 0)

# maping rating 4 and 5 into 1
df_reviews['rating'] = df_reviews['rating'].replace([4, 5], 1)

Just for some use later, we now will create a column that that says that rating = 1 is positive and rating = 0 is negative. 

In [102]:
df_reviews['sentiment'] = df_reviews['rating'].map({1: 'positive', 0: 'negative'})

In [103]:
df_reviews.head()

,unique_id,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime,normalized_review_time,normalized_review,sentiment
1,5957,B002HJV4DE,"[1, 1]",1,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400,2013-10-08,great short read i didnt want to put it down s...,positive
4,1776,B001A06VJ8,"[0, 1]",1,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000,2012-12-31,i did not expect this type of book to be in li...,positive
5,3744,B0021L9YDK,"[6, 6]",1,Aislinn is a little girl with big dreams. Afte...,"12 7, 2009",A3J5NN6MJK4M4A,"Aubrie A. Dionne ""Fantasy, Sci Fi Author""",A story of a little girl with big dreams.,1260144000,2009-12-07,aislinn is a little girl with big dreams after...,positive
6,13641,B0038NN38W,"[1, 1]",0,This has the makings of a good story... unfort...,"08 18, 2011",A531QY5K7JVXI,Chicano,This story has potential but ultimately disapp...,1313625600,2011-08-18,this has the makings of a good story unfortuna...,negative
7,4448,B002AJ7X2C,"[1, 1]",1,I got this because I like collaborated short s...,"03 8, 2010",AN8ELR6AHMMQ,"Jessss ""I read to find stories that inspire m...",Good thriller,1268006400,2010-03-08,i got this because i like collaborated short s...,positive


Now we have a clean and ready-to-go dataframe. 

In [104]:
df_reviews.to_csv('data/dataframes/gold/reviews.csv', index=False)